In [ ]:
# FraudGuard — Feature Engineering & Preprocessing

## Day 3

### Objective

Transform the raw IEEE-CIS Fraud Detection data into a clean, reproducible feature set suitable for machine-learning modeling.

### Today's Goals

- Merge transaction and identity data safely.
- Create meaningful time-based features.
- Engineer missingness and identity-availability features.
- Prepare numerical and categorical features.
- Prevent data leakage during preprocessing.
- Build a reproducible preprocessing pipeline.
- Prepare the dataset for baseline modeling.

> **Important:** Features will be selected based on EDA findings and later validated through model performance. EDA observations are hypotheses, not final feature selections.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"

TRAIN_TRANSACTION = DATA_RAW / "train_transaction.csv"
TRAIN_IDENTITY = DATA_RAW / "train_identity.csv"

print("Project root:", PROJECT_ROOT)
print("Transaction file:", TRAIN_TRANSACTION)
print("Identity file:", TRAIN_IDENTITY)

Project root: /Users/ayushkumar/Desktop/Fraudguard
Transaction file: /Users/ayushkumar/Desktop/Fraudguard/data/raw/train_transaction.csv
Identity file: /Users/ayushkumar/Desktop/Fraudguard/data/raw/train_identity.csv


In [3]:
train_transaction = pd.read_csv(TRAIN_TRANSACTION)

train_identity = pd.read_csv(TRAIN_IDENTITY)

print("Transaction shape:", train_transaction.shape)
print("Identity shape:", train_identity.shape)

Transaction shape: (590540, 394)
Identity shape: (144233, 41)


In [4]:
print(
    "TransactionID unique in transactions:",
    train_transaction["TransactionID"].is_unique
)

print(
    "TransactionID unique in identity:",
    train_identity["TransactionID"].is_unique
)

TransactionID unique in transactions: True
TransactionID unique in identity: True


In [5]:
# Merge transaction and identity data

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print("Master dataset shape:", train.shape)

Master dataset shape: (590540, 434)


In [6]:
print("Original transaction rows:", len(train_transaction))
print("Merged rows:", len(train))
print("Rows preserved:", len(train) == len(train_transaction))

Original transaction rows: 590540
Merged rows: 590540
Rows preserved: True


In [7]:
train["identity_present"] = train["id_01"].notna()

In [8]:
identity_check = (
    train
    .groupby("identity_present")["isFraud"]
    .agg(
        count="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

identity_check["fraud_percentage"] = (
    identity_check["fraud_rate"] * 100
)

display(identity_check)

,count,fraud_count,fraud_rate,fraud_percentage
identity_present,,,,
False,446307,9345,0.020939,2.093850
True,144233,11318,0.078470,7.847025


In [9]:
print(
    "Duplicate TransactionIDs:",
    train["TransactionID"].duplicated().sum()
)

Duplicate TransactionIDs: 0


In [10]:
# Create time-based features from TransactionDT

SECONDS_PER_HOUR = 60 * 60
SECONDS_PER_DAY = 24 * SECONDS_PER_HOUR

train["transaction_hour"] = (
    (train["TransactionDT"] // SECONDS_PER_HOUR) % 24
)

train["transaction_day"] = (
    train["TransactionDT"] // SECONDS_PER_DAY
)

train["transaction_week"] = (
    train["transaction_day"] // 7
)

print("Time features created:")
print([
    "transaction_hour",
    "transaction_day",
    "transaction_week"
])

Time features created:
['transaction_hour', 'transaction_day', 'transaction_week']


In [11]:
train[
    [
        "TransactionDT",
        "transaction_hour",
        "transaction_day",
        "transaction_week",
        "isFraud"
    ]
].head(10)

,TransactionDT,transaction_hour,transaction_day,transaction_week,isFraud
0,86400,0,1,0,0
1,86401,0,1,0,0
2,86469,0,1,0,0
3,86499,0,1,0,0
4,86506,0,1,0,0
5,86510,0,1,0,0
6,86522,0,1,0,0
7,86529,0,1,0,0
8,86535,0,1,0,0
9,86536,0,1,0,0


In [12]:
print(
    "Hour range:",
    train["transaction_hour"].min(),
    "to",
    train["transaction_hour"].max()
)

print(
    "Day range:",
    train["transaction_day"].min(),
    "to",
    train["transaction_day"].max()
)

print(
    "Week range:",
    train["transaction_week"].min(),
    "to",
    train["transaction_week"].max()
)

Hour range: 0 to 23
Day range: 1 to 182
Week range: 0 to 26


In [13]:
hour_fraud = (
    train
    .groupby("transaction_hour")["isFraud"]
    .agg(
        count="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

hour_fraud["fraud_percentage"] = (
    hour_fraud["fraud_rate"] * 100
)

display(hour_fraud)

,count,fraud_count,fraud_rate,fraud_percentage
transaction_hour,,,,
0,37795,1186,0.031380,3.137981
1,32797,1027,0.031314,3.131384
2,26732,1002,0.037483,3.748317
3,20802,797,0.038314,3.831362
4,14839,770,0.051890,5.189029
5,9701,682,0.070302,7.030203
6,6007,467,0.077743,7.774263
7,3704,393,0.106102,10.610151
8,2591,241,0.093014,9.301428


In [14]:
# Encode transaction hour as a cyclical feature

train["hour_sin"] = np.sin(
    2 * np.pi * train["transaction_hour"] / 24
)

train["hour_cos"] = np.cos(
    2 * np.pi * train["transaction_hour"] / 24
)

In [15]:
train[
    [
        "transaction_hour",
        "hour_sin",
        "hour_cos"
    ]
].drop_duplicates(
    subset=["transaction_hour"]
).sort_values(
    "transaction_hour"
)

,transaction_hour,hour_sin,hour_cos
0,0,0.000000e+00,1.000000e+00
228,1,2.588190e-01,9.659258e-01
429,2,5.000000e-01,8.660254e-01
609,3,7.071068e-01,7.071068e-01
743,4,8.660254e-01,5.000000e-01
835,5,9.659258e-01,2.588190e-01
899,6,1.000000e+00,6.123234e-17
990,7,9.659258e-01,-2.588190e-01
1056,8,8.660254e-01,-5.000000e-01
1094,9,7.071068e-01,-7.071068e-01


In [16]:
# Columns used to calculate transaction-level missingness

missingness_features = [
    col for col in train.columns
    if col not in ["TransactionID", "isFraud"]
]

print("Features considered:", len(missingness_features))

Features considered: 438


In [17]:
# Number of missing values in each transaction

train["missing_feature_count"] = (
    train[missingness_features].isna().sum(axis=1)
)

print(
    train["missing_feature_count"].describe()
)

count    590540.000000
mean        195.622774
std          49.035963
min          24.000000
25%         208.000000
50%         211.000000
75%         229.000000
max         340.000000
Name: missing_feature_count, dtype: float64


In [18]:
train["identity_present"] = (
    train["identity_present"]
    .astype(int)
)

print(
    train["identity_present"].value_counts()
)

identity_present
0    446307
1    144233
Name: count, dtype: int64


In [19]:
# Selected missingness indicators identified during EDA

selected_missing_features = [
    "addr1",
    "addr2",
    "M1",
    "M2",
    "M3",
    "M6"
]

for feature in selected_missing_features:
    train[f"{feature}_missing"] = (
        train[feature].isna().astype(int)
    )

print("Missingness indicators created:")
print(
    [f"{feature}_missing" for feature in selected_missing_features]
)

Missingness indicators created:
['addr1_missing', 'addr2_missing', 'M1_missing', 'M2_missing', 'M3_missing', 'M6_missing']


In [20]:
train[
    [
        "missing_feature_count",
        "identity_present",
        "addr1_missing",
        "addr2_missing",
        "M1_missing",
        "M2_missing",
        "M3_missing",
        "M6_missing"
    ]
].head(10)

,missing_feature_count,identity_present,addr1_missing,addr2_missing,M1_missing,M2_missing,M3_missing,M6_missing
0,234,0,0,0,0,0,0,0
1,230,0,0,0,1,1,1,0
2,211,0,0,0,0,0,0,0
3,227,0,0,0,1,1,1,0
4,137,1,0,0,1,1,1,1
5,214,0,0,0,0,0,0,0
6,211,0,0,0,0,0,0,0
7,230,0,0,0,1,1,1,0
8,134,1,0,0,1,1,1,1
9,211,0,0,0,0,0,0,0


In [21]:
print(
    train[
        [
            "missing_feature_count",
            "identity_present"
        ]
    ].describe()
)

       missing_feature_count  identity_present
count          590540.000000     590540.000000
mean              195.622774          0.244239
std                49.035963          0.429636
min                24.000000          0.000000
25%               208.000000          0.000000
50%               211.000000          0.000000
75%               229.000000          0.000000
max               340.000000          1.000000


In [22]:
# Identify categorical columns

categorical_features = train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Number of categorical features:", len(categorical_features))
print(categorical_features)

Number of categorical features: 31
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [23]:
# Check categorical cardinality

categorical_cardinality = (
    train[categorical_features]
    .nunique(dropna=False)
    .sort_values(ascending=False)
)

display(categorical_cardinality)

DeviceInfo       1787
id_33             261
id_31             131
id_30              76
R_emaildomain      61
P_emaildomain      60
card4               5
id_34               5
ProductCD           5
card6               5
id_15               4
M4                  4
id_23               4
M3                  3
DeviceType          3
id_38               3
id_37               3
id_36               3
id_35               3
M1                  3
M2                  3
M5                  3
M6                  3
id_28               3
id_27               3
id_16               3
id_12               3
M9                  3
M8                  3
M7                  3
id_29               3
dtype: int64

In [24]:
# Categorical feature groups

low_cardinality_features = [
    col for col in categorical_features
    if train[col].nunique(dropna=False) <= 10
]

medium_cardinality_features = [
    col for col in categorical_features
    if 10 < train[col].nunique(dropna=False) <= 300
]

high_cardinality_features = [
    col for col in categorical_features
    if train[col].nunique(dropna=False) > 300
]

print("Low cardinality:", low_cardinality_features)
print("\nMedium cardinality:", medium_cardinality_features)
print("\nHigh cardinality:", high_cardinality_features)

Low cardinality: ['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType']

Medium cardinality: ['P_emaildomain', 'R_emaildomain', 'id_30', 'id_31', 'id_33']

High cardinality: ['DeviceInfo']


In [25]:
# Inspect rare-category distribution

for feature in high_cardinality_features + medium_cardinality_features:
    
    counts = train[feature].value_counts(dropna=False)
    
    print(f"\n{feature}")
    print("Total categories:", len(counts))
    print("Categories with < 10 observations:", (counts < 10).sum())
    print("Categories with < 50 observations:", (counts < 50).sum())
    print("Categories with < 100 observations:", (counts < 100).sum())


DeviceInfo
Total categories: 1787
Categories with < 10 observations: 1273
Categories with < 50 observations: 1639
Categories with < 100 observations: 1723

P_emaildomain
Total categories: 60
Categories with < 10 observations: 0
Categories with < 50 observations: 4
Categories with < 100 observations: 8

R_emaildomain
Total categories: 61
Categories with < 10 observations: 2
Categories with < 50 observations: 18
Categories with < 100 observations: 29

id_30
Total categories: 76
Categories with < 10 observations: 4
Categories with < 50 observations: 12
Categories with < 100 observations: 25

id_31
Total categories: 131
Categories with < 10 observations: 28
Categories with < 50 observations: 47
Categories with < 100 observations: 72

id_33
Total categories: 261
Categories with < 10 observations: 181
Categories with < 50 observations: 209
Categories with < 100 observations: 225


In [27]:
RARE_THRESHOLD = 50

def prepare_categorical_features(
    df,
    categorical_columns,
    rare_threshold=RARE_THRESHOLD
):
    """
    Prepare categorical features for encoding.

    - Missing values become __MISSING__
    - Categories occurring fewer than rare_threshold times
      become __RARE__

    The function returns a copy and does not modify the
    original dataframe.
    """

    df = df.copy()

    for feature in categorical_columns:

        # Convert missing values to an explicit category
        df[feature] = df[feature].fillna("__MISSING__")

        # Find category frequencies
        value_counts = df[feature].value_counts()

        # Identify rare categories
        rare_categories = value_counts[
            value_counts < rare_threshold
        ].index

        # Replace rare categories
        df[feature] = df[feature].where(
            ~df[feature].isin(rare_categories),
            "__RARE__"
        )

    return df

In [28]:
train_prepared = prepare_categorical_features(
    train,
    categorical_features
)

In [29]:
print(
    "DeviceInfo categories before:",
    train["DeviceInfo"].nunique(dropna=False)
)

print(
    "DeviceInfo categories after:",
    train_prepared["DeviceInfo"].nunique(dropna=False)
)

DeviceInfo categories before: 1787
DeviceInfo categories after: 149


In [30]:
print(
    train_prepared["DeviceInfo"]
    .value_counts()
    .head(15)
)

DeviceInfo
__MISSING__              471874
Windows                   47722
iOS Device                19782
MacOS                     12573
__RARE__                  11884
Trident/7.0                7440
rv:11.0                    1901
rv:57.0                     962
SM-J700M Build/MMB29K       549
SM-G610M Build/MMB29K       461
SM-G531H Build/LMY48B       410
rv:59.0                     362
SM-G935F Build/NRD90M       334
SM-G955U Build/NRD90M       328
SM-G532M Build/MMB29T       316
Name: count, dtype: int64


In [31]:
# Inspect chronological distribution of transactions

time_summary = (
    train.groupby("transaction_day")["isFraud"]
    .agg(
        count="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

time_summary["fraud_percentage"] = (
    time_summary["fraud_rate"] * 100
)

display(time_summary.head())
display(time_summary.tail())

,count,fraud_count,fraud_rate,fraud_percentage
transaction_day,,,,
1,5122,112,0.021866,2.186646
2,3730,123,0.032976,3.297587
3,3241,92,0.028386,2.838630
4,4036,115,0.028494,2.849356
5,3964,127,0.032038,3.203835


,count,fraud_count,fraud_rate,fraud_percentage
transaction_day,,,,
178,2068,88,0.042553,4.255319
179,2177,66,0.030317,3.031695
180,2618,98,0.037433,3.743316
181,2739,122,0.044542,4.454180
182,2754,119,0.043210,4.320988


In [32]:
print(
    "Earliest transaction day:",
    train["transaction_day"].min()
)

print(
    "Latest transaction day:",
    train["transaction_day"].max()
)

print(
    "Total encoded days:",
    train["transaction_day"].nunique()
)

Earliest transaction day: 1
Latest transaction day: 182
Total encoded days: 182


In [33]:
# Inspect transaction volume and fraud rate across the timeline

time_summary = (
    train.groupby("transaction_day")["isFraud"]
    .agg(
        count="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

time_summary["fraud_percentage"] = (
    time_summary["fraud_rate"] * 100
)

display(time_summary.head(10))
display(time_summary.tail(10))

,count,fraud_count,fraud_rate,fraud_percentage
transaction_day,,,,
1,5122,112,0.021866,2.186646
2,3730,123,0.032976,3.297587
3,3241,92,0.028386,2.838630
4,4036,115,0.028494,2.849356
5,3964,127,0.032038,3.203835
6,3717,99,0.026634,2.663438
7,3786,136,0.035922,3.592182
8,3999,93,0.023256,2.325581
9,3907,119,0.030458,3.045815


,count,fraud_count,fraud_rate,fraud_percentage
transaction_day,,,,
173,2866,142,0.049546,4.954641
174,2837,87,0.030666,3.066620
175,2683,108,0.040253,4.025345
176,3032,145,0.047823,4.782322
177,2364,109,0.046108,4.610829
178,2068,88,0.042553,4.255319
179,2177,66,0.030317,3.031695
180,2618,98,0.037433,3.743316
181,2739,122,0.044542,4.454180


In [34]:
# Compare fraud distribution across early, middle, and late periods

train["time_period"] = pd.cut(
    train["transaction_day"],
    bins=[0, 60, 120, 182],
    labels=["Early", "Middle", "Late"]
)

period_summary = (
    train.groupby("time_period", observed=True)["isFraud"]
    .agg(
        count="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

period_summary["fraud_percentage"] = (
    period_summary["fraud_rate"] * 100
)

display(period_summary)

,count,fraud_count,fraud_rate,fraud_percentage
time_period,,,,
Early,223738,6978,0.031188,3.118826
Middle,190804,7622,0.039947,3.994675
Late,175998,6063,0.034449,3.444926


In [35]:
# Time-based train/validation split

TRAIN_END_DAY = 145

train_data = train[
    train["transaction_day"] <= TRAIN_END_DAY
].copy()

validation_data = train[
    train["transaction_day"] > TRAIN_END_DAY
].copy()

print("Training shape:", train_data.shape)
print("Validation shape:", validation_data.shape)

print(
    "\nTraining day range:",
    train_data["transaction_day"].min(),
    "to",
    train_data["transaction_day"].max()
)

print(
    "Validation day range:",
    validation_data["transaction_day"].min(),
    "to",
    validation_data["transaction_day"].max()
)

Training shape: (484847, 448)
Validation shape: (105693, 448)

Training day range: 1 to 145
Validation day range: 146 to 182


In [36]:
# Verify target distribution in both splits

split_summary = pd.DataFrame({
    "train": [
        len(train_data),
        train_data["isFraud"].sum(),
        train_data["isFraud"].mean() * 100
    ],
    "validation": [
        len(validation_data),
        validation_data["isFraud"].sum(),
        validation_data["isFraud"].mean() * 100
    ]
}, index=[
    "transactions",
    "fraud_count",
    "fraud_percentage"
])

display(split_summary)

,train,validation
transactions,484847.000000,105693.000000
fraud_count,17052.000000,3611.000000
fraud_percentage,3.516986,3.416499


In [37]:
# Separate features and target

TARGET = "isFraud"
ID_COLUMN = "TransactionID"

X_train = train_data.drop(
    columns=[TARGET, ID_COLUMN]
)

y_train = train_data[TARGET]

X_validation = validation_data.drop(
    columns=[TARGET, ID_COLUMN]
)

y_validation = validation_data[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

X_train: (484847, 446)
y_train: (484847,)
X_validation: (105693, 446)
y_validation: (105693,)


In [38]:
print("Target in X_train:", TARGET in X_train.columns)
print("TransactionID in X_train:", ID_COLUMN in X_train.columns)

Target in X_train: False
TransactionID in X_train: False


In [39]:
# Identify feature types from the training data

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", len(categorical_features))
print(categorical_features)

print("\nNumerical features:", len(numerical_features))
print(numerical_features[:20])

Categorical features: 32
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo', 'time_period']

Numerical features: 414
['TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10']


In [40]:
other_features = [
    col for col in X_train.columns
    if col not in categorical_features + numerical_features
]

print("\nOther feature types:", other_features)


Other feature types: []


In [1]:
from sklearn.impute import SimpleImputer

numerical_imputer = SimpleImputer(
    strategy="median"
)

X_train_num = numerical_imputer.fit_transform(
    X_train[numerical_features]
)

X_validation_num = numerical_imputer.transform(
    X_validation[numerical_features]
)

print("Training numerical matrix:", X_train_num.shape)
print("Validation numerical matrix:", X_validation_num.shape)

NameError: name 'X_train' is not defined

In [ ]:
print(
    "Missing values after training imputation:",
    np.isnan(X_train_num).sum()
)

print(
    "Missing values after validation imputation:",
    np.isnan(X_validation_num).sum()
)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

TRAIN_TRANSACTION = DATA_RAW / "train_transaction.csv"
TRAIN_IDENTITY = DATA_RAW / "train_identity.csv"

train_transaction = pd.read_csv(TRAIN_TRANSACTION)
train_identity = pd.read_csv(TRAIN_IDENTITY)

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(train.shape)

(590540, 434)


In [3]:
SECONDS_PER_HOUR = 60 * 60
SECONDS_PER_DAY = 24 * SECONDS_PER_HOUR

train["transaction_hour"] = (
    (train["TransactionDT"] // SECONDS_PER_HOUR) % 24
)

train["transaction_day"] = (
    train["TransactionDT"] // SECONDS_PER_DAY
)

train["transaction_week"] = (
    train["transaction_day"] // 7
)

train["hour_sin"] = np.sin(
    2 * np.pi * train["transaction_hour"] / 24
)

train["hour_cos"] = np.cos(
    2 * np.pi * train["transaction_hour"] / 24
)

train["identity_present"] = (
    train["id_01"].notna().astype(int)
)

In [4]:
missingness_features = [
    col for col in train.columns
    if col not in ["TransactionID", "isFraud"]
]

train["missing_feature_count"] = (
    train[missingness_features].isna().sum(axis=1)
)

selected_missing_features = [
    "addr1",
    "addr2",
    "M1",
    "M2",
    "M3",
    "M6"
]

for feature in selected_missing_features:
    train[f"{feature}_missing"] = (
        train[feature].isna().astype(int)
    )

In [5]:
train_data = train[
    train["transaction_day"] <= 145
].copy()

validation_data = train[
    train["transaction_day"] > 145
].copy()

TARGET = "isFraud"
ID_COLUMN = "TransactionID"

X_train = train_data.drop(
    columns=[TARGET, ID_COLUMN]
)

y_train = train_data[TARGET]

X_validation = validation_data.drop(
    columns=[TARGET, ID_COLUMN]
)

y_validation = validation_data[TARGET]

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)

X_train: (484847, 445)
X_validation: (105693, 445)


In [6]:
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical:", len(categorical_features))
print("Numerical:", len(numerical_features))

Categorical: 31
Numerical: 414


In [7]:
numerical_imputer = SimpleImputer(
    strategy="median"
)

X_train_num = numerical_imputer.fit_transform(
    X_train[numerical_features]
)

X_validation_num = numerical_imputer.transform(
    X_validation[numerical_features]
)

print("Training numerical matrix:", X_train_num.shape)
print("Validation numerical matrix:", X_validation_num.shape)

Training numerical matrix: (484847, 414)
Validation numerical matrix: (105693, 414)


In [8]:
print(
    "Missing values after training imputation:",
    np.isnan(X_train_num).sum()
)

print(
    "Missing values after validation imputation:",
    np.isnan(X_validation_num).sum()
)

Missing values after training imputation: 0
Missing values after validation imputation: 0


In [9]:
from sklearn.preprocessing import OneHotEncoder

RARE_THRESHOLD = 50

categorical_maps = {}

for feature in categorical_features:
    values = X_train[feature].fillna("__MISSING__")
    
    counts = values.value_counts()
    
    frequent_categories = counts[
        counts >= RARE_THRESHOLD
    ].index
    
    categorical_maps[feature] = set(
        frequent_categories
    )

print("Categorical mappings created:", len(categorical_maps))

Categorical mappings created: 31


In [10]:
def transform_categorical(
    df,
    categorical_columns,
    categorical_maps
):
    df = df[categorical_columns].copy()

    for feature in categorical_columns:
        df[feature] = df[feature].fillna("__MISSING__")

        frequent_categories = categorical_maps[feature]

        df[feature] = df[feature].where(
            df[feature].isin(frequent_categories),
            "__RARE__"
        )

    return df

In [11]:
X_train_cat = transform_categorical(
    X_train,
    categorical_features,
    categorical_maps
)

X_validation_cat = transform_categorical(
    X_validation,
    categorical_features,
    categorical_maps
)

In [12]:
print(
    "Training categorical shape:",
    X_train_cat.shape
)

print(
    "Validation categorical shape:",
    X_validation_cat.shape
)

Training categorical shape: (484847, 31)
Validation categorical shape: (105693, 31)


In [13]:
from sklearn.preprocessing import OneHotEncoder

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

X_train_cat_encoded = categorical_encoder.fit_transform(
    X_train_cat
)

X_validation_cat_encoded = categorical_encoder.transform(
    X_validation_cat
)

print(
    "Encoded training shape:",
    X_train_cat_encoded.shape
)

print(
    "Encoded validation shape:",
    X_validation_cat_encoded.shape
)

Encoded training shape: (484847, 477)
Encoded validation shape: (105693, 477)


In [14]:
print(
    "Number of one-hot features:",
    X_train_cat_encoded.shape[1]
)

Number of one-hot features: 477


In [15]:
feature_names = (
    categorical_encoder
    .get_feature_names_out(categorical_features)
)

print("Total feature names:", len(feature_names))

print(feature_names[:30])

Total feature names: 477
['ProductCD_C' 'ProductCD_H' 'ProductCD_R' 'ProductCD_S' 'ProductCD_W'
 'card4___MISSING__' 'card4_american express' 'card4_discover'
 'card4_mastercard' 'card4_visa' 'card6___MISSING__' 'card6___RARE__'
 'card6_credit' 'card6_debit' 'P_emaildomain___MISSING__'
 'P_emaildomain___RARE__' 'P_emaildomain_aim.com'
 'P_emaildomain_anonymous.com' 'P_emaildomain_aol.com'
 'P_emaildomain_att.net' 'P_emaildomain_bellsouth.net'
 'P_emaildomain_cableone.net' 'P_emaildomain_centurylink.net'
 'P_emaildomain_cfl.rr.com' 'P_emaildomain_charter.net'
 'P_emaildomain_comcast.net' 'P_emaildomain_cox.net'
 'P_emaildomain_earthlink.net' 'P_emaildomain_embarqmail.com'
 'P_emaildomain_frontier.com']


In [16]:
from scipy.sparse import hstack

In [17]:
X_train_final = hstack([
    X_train_num,
    X_train_cat_encoded
]).tocsr()

X_validation_final = hstack([
    X_validation_num,
    X_validation_cat_encoded
]).tocsr()

In [18]:
print(
    "Final training shape:",
    X_train_final.shape
)

print(
    "Final validation shape:",
    X_validation_final.shape
)

Final training shape: (484847, 891)
Final validation shape: (105693, 891)


In [19]:
print("Training matrix type:", type(X_train_final))
print("Validation matrix type:", type(X_validation_final))

print(
    "Training non-zero values:",
    X_train_final.nnz
)

Training matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Validation matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Training non-zero values: 108839426


In [20]:
print("Final training shape:", X_train_final.shape)
print("Final validation shape:", X_validation_final.shape)

print("Training fraud rate:", y_train.mean() * 100)
print("Validation fraud rate:", y_validation.mean() * 100)

Final training shape: (484847, 891)
Final validation shape: (105693, 891)
Training fraud rate: 3.5169857707689234
Validation fraud rate: 3.416498727446472


In [21]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(
    max_iter=100,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

baseline_model.fit(
    X_train_final,
    y_train
)

/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LogisticRegression(class_weight='balanced', random_state=42, solver='liblinear')

In [22]:
print("Baseline model trained successfully.")

Baseline model trained successfully.


In [24]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(
    max_iter=500,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

baseline_model.fit(
    X_train_final,
    y_train
)

print("Baseline model trained successfully.")

Baseline model trained successfully.


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [25]:
y_validation_proba = baseline_model.predict_proba(
    X_validation_final
)[:, 1]

print(y_validation_proba[:10])

[0.45664229 0.60039098 0.57403996 0.43447433 0.72119724 0.41019264
 0.13499347 0.43039207 0.4960234  0.77520302]


In [26]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

roc_auc = roc_auc_score(
    y_validation,
    y_validation_proba
)

pr_auc = average_precision_score(
    y_validation,
    y_validation_proba
)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")

ROC-AUC: 0.8054
PR-AUC:  0.1622


In [27]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

threshold_results = []

for threshold in thresholds:

    y_pred = (
        y_validation_proba >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            y_pred,
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(
    threshold_results
)

display(threshold_results)

,threshold,precision,recall,f1
0,0.1,0.035946,0.995292,0.069387
1,0.2,0.039389,0.973415,0.075714
2,0.3,0.044335,0.953752,0.084732
3,0.4,0.053433,0.889227,0.100808
4,0.5,0.089899,0.734976,0.160203
5,0.6,0.128246,0.603157,0.211518
6,0.7,0.159619,0.510108,0.243152
7,0.8,0.233875,0.362503,0.284318
8,0.9,0.296149,0.185267,0.227939


In [28]:
threshold = 0.8

y_pred_08 = (
    y_validation_proba >= threshold
).astype(int)

from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(
    y_validation,
    y_pred_08
).ravel()

print("Threshold:", threshold)
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives  :", tp)

print("\nTotal flagged:", fp + tp)
print("Fraud caught:", tp)
print("Fraud missed:", fn)

Threshold: 0.8
True Negatives : 97794
False Positives: 4288
False Negatives: 2302
True Positives  : 1309

Total flagged: 5597
Fraud caught: 1309
Fraud missed: 2302


In [ ]:
## 📅 Day 3 — Feature Engineering & Baseline Modeling

- Merged transaction and identity data using a safe left join and engineered time, identity-availability, and missingness features.
- Created a leakage-safe chronological split: 484,847 training and 105,693 validation transactions.
- Built numerical median imputation and categorical rare-category grouping with one-hot encoding, producing 891 final features.
- Trained a Logistic Regression baseline with class balancing, achieving ROC-AUC 0.8054 and PR-AUC 0.1622.
- At threshold 0.80, the baseline caught 1,309 of 3,611 frauds with 23.39% precision and 36.25% recall.
- Established the baseline benchmark; future models and feature improvements must outperform these results.

In [1]:
feature_names = np.concatenate([
    numerical_features,
    categorical_encoder.get_feature_names_out(
        categorical_features
    )
])

print("Total feature names:", len(feature_names))

pd.DataFrame({
    "feature": feature_names
}).to_csv(
    processed_dir / "feature_names.csv",
    index=False
)

print("Feature names saved successfully.")

NameError: name 'np' is not defined

In [2]:
import numpy as np
import pandas as pd

In [3]:
feature_names = np.concatenate([
    numerical_features,
    categorical_encoder.get_feature_names_out(
        categorical_features
    )
])

print("Total feature names:", len(feature_names))

pd.DataFrame({
    "feature": feature_names
}).to_csv(
    processed_dir / "feature_names.csv",
    index=False
)

print("Feature names saved successfully.")

NameError: name 'numerical_features' is not defined